# Lab: harden your MCP server, with an agent in the loop

A real agent (DeepSeek, through OpenRouter) uses a music server while the network misbehaves: **one read in four is rate-limited**, and **the first of every three writes is executed but its response is lost**. You watch it fail on the naive server, fill in **five type annotations** in `my_mcp_server.py`, read the **dispatcher** that retries only when it is safe, and watch the agent again.

**Setup:** put `OPENROUTER_API_KEY=...` in a `.env` file (see `.env.example`); never commit it. Run the cells in order.

In [ ]:
import hashlib, json
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_random_exponential
from lab import check, check_server, trials, typed_error   # given: runs the agent against your server (see lab.py if curious)

## 1. Watch the agent on the naive server (v0.1)

Three runs of the same task on `naive_server.py`. The table is what the server **really did**; "agent says" is what the user is told. The model is not deterministic: runs differ.

In [ ]:
before = await trials(server_file="naive_server.py")
before

## 2. Fill in the types of your server

Open `my_mcp_server.py`. The typed errors, the `idempotency_key` argument and the tool bodies are given. Fill in the five lines marked TODO:

1. **`READ` and `WRITE` annotations**: which tool is read-only, which one is safe to repeat, and what makes it safe?
2. **`TrackId`, `Playlist`, `Query`**: `Annotated[str, Field(pattern=..., description=...)]` and `Literal[...]`. The model sees the schema built from them.

The next cell tells you what is still missing.

In [ ]:
await check_server()

## 3. Read the dispatcher (given)

The **harness**, not the model, computes the key from `(run_id, tool, arguments)`, once per logical call. It retries only if the error is `retryable` **and** repeating is safe (read-only, or keyed), with backoff and jitter from `tenacity`. Read the code, then run it.

In [ ]:
def idempotency_key(run_id, tool, args):
    """Same logical call -> same key, at every attempt: no timestamp, no random value."""
    payload = json.dumps([run_id, tool, args], sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


class Retryable(Exception):
    def __init__(self, result): self.result = result


async def call_with_retries(call, tool, args, run_id, keyed, read_only):
    # `call(args)` calls the tool and returns its result. `keyed`: the tool accepts an idempotency_key.
    if keyed:                                                  # computed ONCE, before the first attempt
        args = {**args, "idempotency_key": idempotency_key(run_id, tool, args)}

    @retry(retry=retry_if_exception_type(Retryable), stop=stop_after_attempt(3),
           wait=wait_random_exponential(multiplier=0.2, max=2),
           retry_error_callback=lambda state: state.outcome.exception().result)
    async def attempt():
        result = await call(args)
        error = typed_error(result)
        if error and error.get("retryable") and (read_only or keyed):    # safe to repeat
            raise Retryable(result)
        return result                                          # otherwise the model reads the error

    return await attempt()

In [ ]:
await check(call_with_retries)             # no model: the same write twice must leave one change

## 4. Watch the agent again (v0.2)

Same task, same chaos, your server and your dispatcher. Why must the **harness**, not the model, compute the key? What did the agent see when a write timed out, before and after?

In [ ]:
after = await trials(call_with_retries)
after